In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import os

BASE = "/content/drive/MyDrive/iot/iot device name/network"

print("BASE exists?", os.path.exists(BASE))
print(os.listdir(BASE)[:20])

BASE exists? True
['normal_1.pcap', 'normal_4.pcap', 'normal_5.pcap', 'normal_6.pcap', 'normal_7.pcap', 'normal_8.pcap', 'normal_9.pcap', 'normal_10.pcap', 'normal_11.pcap', 'normal_12.pcap', 'normal_13.pcap', 'normal_3.pcap', 'normal_2.pcap', 'packet_csvs_labeled', 'packet_csvs_device_only']


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob

BASE = "/content/drive/MyDrive/iot/iot device name/network"
OKUI22_DIR = os.path.join(BASE, "Okui22")
PACKETS_RAW_DIR = os.path.join(OKUI22_DIR, "packets_raw")
FLOW_DIR = os.path.join(OKUI22_DIR, "flow_records")
FEATURE_DIR = os.path.join(OKUI22_DIR, "features")

for d in [OKUI22_DIR, PACKETS_RAW_DIR, FLOW_DIR, FEATURE_DIR]:
    os.makedirs(d, exist_ok=True)

pcap_files = sorted(glob.glob(os.path.join(BASE, "normal_*.pcap")))

print("BASE exists:", os.path.exists(BASE))
print("num pcaps:", len(pcap_files))
print("first few pcaps:", [os.path.basename(x) for x in pcap_files[:5]])
print("Okui22 contents:", os.listdir(OKUI22_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE exists: True
num pcaps: 13
first few pcaps: ['normal_1.pcap', 'normal_10.pcap', 'normal_11.pcap', 'normal_12.pcap', 'normal_13.pcap']
Okui22 contents: ['packets_raw', 'flow_records', 'features']


In [4]:
!apt-get -qq update
!DEBIAN_FRONTEND=noninteractive apt-get -y -qq install tshark
!tshark -v | head -n 3

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package libpcap0.8:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../00-libpcap0.8_1.10.1-4ubuntu1.22.04.1_amd64.deb ...
Unpacking libpcap0.8:amd64 (1.10.1-4ubuntu1.22.04.1) ...
Selecting previously unselected package libbcg729-0:amd64.
Preparing to unpack .../01-libbcg729-0_1.1.1-2_amd64.deb ...
Unpacking libbcg729-0:amd64 (1.1.1-2) ...
Selecting previously unselected package liblua5.2-0:amd64.
Preparing to unpack .../02-liblua5.2-0_5.2.4-2_amd64.deb ...
Unpacking liblua5.2-0:amd64 (5.2.4-2) ...
Selecting previously unselected package libnl-genl-3-200:amd64.
Preparing to unpack .../03-libnl-genl-3-200_3.5.0-0.1_amd64.deb ...
Unpacking libnl-genl-3-200:amd64 (3.5.0-0.1) ...
Selecting prev

In [7]:
import os
import subprocess

fields = [
    "frame.time_epoch",
    "ip.src",
    "ip.dst",
    "ip.proto",
    "tcp.srcport",
    "tcp.dstport",
    "udp.srcport",
    "udp.dstport",
    "ip.len",
    "frame.len",
]

for pcap in pcap_files:
    out_csv = os.path.join(
        PACKETS_RAW_DIR,
        os.path.basename(pcap).replace(".pcap", "_packets.csv")
    )

    if os.path.exists(out_csv):
        print("skip existing:", os.path.basename(out_csv))
        continue

    cmd = [
        "tshark",
        "-n",
        "-r", pcap,
        "-Y", "ip",
        "-T", "fields",
        "-E", "header=y",
        "-E", "separator=,",
        "-E", "quote=d",
    ]

    for f in fields:
        cmd.extend(["-e", f])

    print("extracting:", os.path.basename(pcap))
    with open(out_csv, "w", encoding="utf-8", newline="") as fout:
        result = subprocess.run(cmd, stdout=fout, stderr=subprocess.PIPE, text=True)

    if result.returncode != 0:
        print("ERROR on", os.path.basename(pcap))
        print(result.stderr[:1000])
        break
    else:
        print("saved:", out_csv)

print("done")

extracting: normal_1.pcap
saved: /content/drive/MyDrive/iot/iot device name/network/Okui22/packets_raw/normal_1_packets.csv
extracting: normal_10.pcap
saved: /content/drive/MyDrive/iot/iot device name/network/Okui22/packets_raw/normal_10_packets.csv
extracting: normal_11.pcap
saved: /content/drive/MyDrive/iot/iot device name/network/Okui22/packets_raw/normal_11_packets.csv
extracting: normal_12.pcap
saved: /content/drive/MyDrive/iot/iot device name/network/Okui22/packets_raw/normal_12_packets.csv
extracting: normal_13.pcap
saved: /content/drive/MyDrive/iot/iot device name/network/Okui22/packets_raw/normal_13_packets.csv
extracting: normal_2.pcap
saved: /content/drive/MyDrive/iot/iot device name/network/Okui22/packets_raw/normal_2_packets.csv
extracting: normal_3.pcap
saved: /content/drive/MyDrive/iot/iot device name/network/Okui22/packets_raw/normal_3_packets.csv
extracting: normal_4.pcap
saved: /content/drive/MyDrive/iot/iot device name/network/Okui22/packets_raw/normal_4_packets.csv


In [9]:
import pandas as pd
import os

BASE = "/content/drive/MyDrive/iot/iot device name/network"
OKUI22_DIR = os.path.join(BASE, "Okui22")
os.makedirs(OKUI22_DIR, exist_ok=True)

device_map = pd.DataFrame({
    "ip": [
        "192.168.1.133",
        "192.168.1.146",
        "192.168.1.152",
        "192.168.1.17",
        "192.168.1.190",
        "192.168.1.191",
        "192.168.1.192",
        "192.168.1.193",
        "192.168.1.194",
        "192.168.1.250",
        "192.168.1.30",
        "192.168.1.79",
    ],
    "best_mac": [
        "dc:56:e7:5b:61:41",
        "60:14:b3:b1:91:73",
        "00:0c:29:d2:b0:02",
        "80:3f:5d:10:17:e1",
        "00:0c:29:ee:e0:7a",
        "80:3f:5d:10:17:e1",
        "60:14:b3:b1:91:73",
        "60:14:b3:b1:91:73",
        "60:14:b3:b1:91:73",
        "d4:dc:cd:b4:26:3e",
        "80:3f:5d:10:17:e1",
        "00:c3:f4:0f:67:73",
    ],
    "service_name": [
        "service_7",
        "service_2",
        "service_1",
        "service_5",
        "service_3",
        "service_5",
        "service_2",
        "service_2",
        "service_2",
        "service_6",
        "service_5",
        "service_4",
    ]
})

map_path = os.path.join(OKUI22_DIR, "device_map.csv")
device_map.to_csv(map_path, index=False)
print("saved:", map_path)
device_map

saved: /content/drive/MyDrive/iot/iot device name/network/Okui22/device_map.csv


,ip,best_mac,service_name
0,192.168.1.133,dc:56:e7:5b:61:41,service_7
1,192.168.1.146,60:14:b3:b1:91:73,service_2
2,192.168.1.152,00:0c:29:d2:b0:02,service_1
3,192.168.1.17,80:3f:5d:10:17:e1,service_5
4,192.168.1.190,00:0c:29:ee:e0:7a,service_3
5,192.168.1.191,80:3f:5d:10:17:e1,service_5
6,192.168.1.192,60:14:b3:b1:91:73,service_2
7,192.168.1.193,60:14:b3:b1:91:73,service_2
8,192.168.1.194,60:14:b3:b1:91:73,service_2
9,192.168.1.250,d4:dc:cd:b4:26:3e,service_6


In [10]:
m = pd.read_csv(map_path)

print("unique IPs:", m["ip"].nunique())
print("unique MACs:", m["best_mac"].nunique())
print("unique service names:", m["service_name"].nunique())

print("\nMAC -> number of service_name")
display(m.groupby("best_mac")["service_name"].nunique().reset_index(name="n_service_names"))

print("\nservice_name -> number of MACs")
display(m.groupby("service_name")["best_mac"].nunique().reset_index(name="n_macs"))

unique IPs: 12
unique MACs: 7
unique service names: 7

MAC -> number of service_name


,best_mac,n_service_names
0,00:0c:29:d2:b0:02,1
1,00:0c:29:ee:e0:7a,1
2,00:c3:f4:0f:67:73,1
3,60:14:b3:b1:91:73,1
4,80:3f:5d:10:17:e1,1
5,d4:dc:cd:b4:26:3e,1
6,dc:56:e7:5b:61:41,1



service_name -> number of MACs


,service_name,n_macs
0,service_1,1
1,service_2,1
2,service_3,1
3,service_4,1
4,service_5,1
5,service_6,1
6,service_7,1


In [11]:
import pandas as pd
import numpy as np
import os
import glob

BASE = "/content/drive/MyDrive/iot/iot device name/network"
OKUI22_DIR = os.path.join(BASE, "Okui22")
PACKETS_RAW_DIR = os.path.join(OKUI22_DIR, "packets_raw")
FLOW_DIR = os.path.join(OKUI22_DIR, "flow_records")
FEATURE_DIR = os.path.join(OKUI22_DIR, "features")

os.makedirs(FLOW_DIR, exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

map_path = os.path.join(OKUI22_DIR, "device_map.csv")
devmap = pd.read_csv(map_path)

devmap["ip"] = devmap["ip"].astype(str).str.strip()
devmap["best_mac"] = devmap["best_mac"].astype(str).str.strip().str.lower()
devmap["service_name"] = devmap["service_name"].astype(str).str.strip()

ip_to_mac = dict(zip(devmap["ip"], devmap["best_mac"]))
ip_to_name = dict(zip(devmap["ip"], devmap["service_name"]))

known_ips = set(devmap["ip"])
print("known IPs:", len(known_ips))
print("known MACs:", devmap["best_mac"].nunique())
print("known names:", devmap["service_name"].nunique())

known IPs: 12
known MACs: 7
known names: 7


In [12]:
packet_files = sorted(glob.glob(os.path.join(PACKETS_RAW_DIR, "*_packets.csv")))
WINDOW = 1800.0

all_features = []

for f in packet_files:
    print("processing:", os.path.basename(f))
    pkt = pd.read_csv(f)
    pkt.columns = [c.strip() for c in pkt.columns]

    num_cols = [
        "frame.time_epoch",
        "ip.proto",
        "tcp.srcport", "tcp.dstport",
        "udp.srcport", "udp.dstport",
        "ip.len", "frame.len"
    ]
    for c in num_cols:
        if c in pkt.columns:
            pkt[c] = pd.to_numeric(pkt[c], errors="coerce")

    pkt["ip.src"] = pkt["ip.src"].astype(str).str.strip()
    pkt["ip.dst"] = pkt["ip.dst"].astype(str).str.strip()

    pkt["src_port"] = pkt["tcp.srcport"].fillna(pkt["udp.srcport"])
    pkt["dst_port"] = pkt["tcp.dstport"].fillna(pkt["udp.dstport"])
    pkt["bytes_ip"] = pkt["ip.len"].fillna(pkt["frame.len"])

    pkt = pkt.dropna(subset=["frame.time_epoch", "ip.src", "ip.dst", "ip.proto", "bytes_ip"]).copy()
    pkt = pkt[(pkt["ip.src"] != "nan") & (pkt["ip.dst"] != "nan")]

    # 只保留和已知设备 IP 有关的包
    mask = pkt["ip.src"].isin(known_ips) | pkt["ip.dst"].isin(known_ips)
    pkt = pkt.loc[mask].copy()

    if len(pkt) == 0:
        print("  no mapped packets, skip")
        continue

    # 从设备视角定义方向
    pkt["device_ip"] = np.where(pkt["ip.src"].isin(known_ips), pkt["ip.src"], pkt["ip.dst"])
    pkt["device_id"] = pkt["device_ip"].map(ip_to_mac)
    pkt["device_name"] = pkt["device_ip"].map(ip_to_name)

    pkt["direction"] = np.where(pkt["ip.src"] == pkt["device_ip"], "fwd", "rev")
    pkt["remote_ip"] = np.where(pkt["direction"] == "fwd", pkt["ip.dst"], pkt["ip.src"])
    pkt["local_port"] = np.where(pkt["direction"] == "fwd", pkt["src_port"], pkt["dst_port"])
    pkt["remote_port"] = np.where(pkt["direction"] == "fwd", pkt["dst_port"], pkt["src_port"])

    pkt = pkt.dropna(subset=["device_id", "device_name", "local_port", "remote_port"]).copy()
    pkt["local_port"] = pkt["local_port"].astype(int)
    pkt["remote_port"] = pkt["remote_port"].astype(int)
    pkt["proto"] = pkt["ip.proto"].astype(int)

    pkt["fwd_bytes"] = np.where(pkt["direction"] == "fwd", pkt["bytes_ip"], 0)
    pkt["rev_bytes"] = np.where(pkt["direction"] == "rev", pkt["bytes_ip"], 0)
    pkt["fwd_pkts"] = np.where(pkt["direction"] == "fwd", 1, 0)
    pkt["rev_pkts"] = np.where(pkt["direction"] == "rev", 1, 0)

    flow = (
        pkt.groupby(
            ["device_id", "device_name", "device_ip", "remote_ip", "local_port", "remote_port", "proto"],
            as_index=False
        )
        .agg(
            flow_start=("frame.time_epoch", "min"),
            flow_end=("frame.time_epoch", "max"),
            octetTotalCount=("fwd_bytes", "sum"),
            reverseOctetTotalCount=("rev_bytes", "sum"),
            packetTotalCount=("fwd_pkts", "sum"),
            reversePacketTotalCount=("rev_pkts", "sum"),
        )
    )

    flow["duration"] = flow["flow_end"] - flow["flow_start"]

    flow_out = os.path.join(
        FLOW_DIR,
        os.path.basename(f).replace("_packets.csv", "_flows.csv")
    )
    flow.to_csv(flow_out, index=False)

    # 1800 秒窗口
    flow["window_start"] = (flow["flow_end"] // WINDOW) * WINDOW

    agg_dict = {"record_count": ("remote_ip", "size")}
    for m in [
        "octetTotalCount",
        "reverseOctetTotalCount",
        "packetTotalCount",
        "reversePacketTotalCount",
    ]:
        agg_dict[f"{m}_max"] = (m, "max")
        agg_dict[f"{m}_min"] = (m, "min")
        agg_dict[f"{m}_mean"] = (m, "mean")
        agg_dict[f"{m}_median"] = (m, "median")

    feat = (
        flow.groupby(["device_id", "device_name", "window_start"], as_index=False)
            .agg(**agg_dict)
            .fillna(0)
    )

    feat["source_file"] = os.path.basename(f)
    feat = feat[
        ["source_file", "device_id", "device_name", "window_start"] +
        [c for c in feat.columns if c not in ["source_file", "device_id", "device_name", "window_start"]]
    ]

    feat_out = os.path.join(
        FEATURE_DIR,
        os.path.basename(f).replace("_packets.csv", "_okui22_features.csv")
    )
    feat.to_csv(feat_out, index=False)
    all_features.append(feat)

    print("  saved flow:", os.path.basename(flow_out))
    print("  saved feat:", os.path.basename(feat_out))

if all_features:
    final_features = pd.concat(all_features, ignore_index=True)
    final_out = os.path.join(FEATURE_DIR, "okui22_window_features_all.csv")
    final_features.to_csv(final_out, index=False)
    print("final saved:", final_out)
    print("final shape:", final_features.shape)
    display(final_features.head())
else:
    print("no features generated")

processing: normal_10_packets.csv


/tmp/ipykernel_8127/2165278163.py:8: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  pkt = pd.read_csv(f)


  saved flow: normal_10_flows.csv
  saved feat: normal_10_okui22_features.csv
processing: normal_11_packets.csv
  saved flow: normal_11_flows.csv
  saved feat: normal_11_okui22_features.csv
processing: normal_12_packets.csv


/tmp/ipykernel_8127/2165278163.py:8: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  pkt = pd.read_csv(f)


  saved flow: normal_12_flows.csv
  saved feat: normal_12_okui22_features.csv
processing: normal_13_packets.csv


...

/tmp/ipykernel_8127/2165278163.py:8: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  pkt = pd.read_csv(f)


  saved flow: normal_8_flows.csv
  saved feat: normal_8_okui22_features.csv
processing: normal_9_packets.csv


/tmp/ipykernel_8127/2165278163.py:8: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  pkt = pd.read_csv(f)


  saved flow: normal_9_flows.csv
  saved feat: normal_9_okui22_features.csv
final saved: /content/drive/MyDrive/iot/iot device name/network/Okui22/features/okui22_window_features_all.csv
final shape: (367, 21)


,source_file,device_id,device_name,window_start,record_count,octetTotalCount_max,octetTotalCount_min,octetTotalCount_mean,octetTotalCount_median,reverseOctetTotalCount_max,...,reverseOctetTotalCount_mean,reverseOctetTotalCount_median,packetTotalCount_max,packetTotalCount_min,packetTotalCount_mean,packetTotalCount_median,reversePacketTotalCount_max,reversePacketTotalCount_min,reversePacketTotalCount_mean,reversePacketTotalCount_median
0,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554323e+09,94,1240.0,70.0,197.957447,186.0,1232.0,...,20.436170,0.0,18,1,2.223404,2.0,16,0,0.244681,0.0
1,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554325e+09,138,17089.0,62.0,354.891304,186.0,47043.0,...,460.376812,0.0,37,1,2.673913,2.0,35,0,0.739130,0.0
2,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554327e+09,128,2750.0,70.0,209.570312,186.0,4221.0,...,45.492188,0.0,21,1,2.242188,2.0,18,0,0.281250,0.0
3,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554329e+09,123,2700.0,73.0,222.560976,186.0,4371.0,...,68.406504,0.0,20,1,2.260163,2.0,16,0,0.235772,0.0
4,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554331e+09,127,2700.0,73.0,220.803150,186.0,5687.0,...,70.330709,0.0,22,1,2.291339,2.0,20,0,0.283465,0.0


In [13]:
final_out = os.path.join(FEATURE_DIR, "okui22_window_features_all.csv")
df = pd.read_csv(final_out)

print(df.columns.tolist())
print(df.shape)
print("\nSamples per device_id:")
print(df["device_id"].value_counts())

print("\nSamples per device_name:")
print(df["device_name"].value_counts())

df.head()

['source_file', 'device_id', 'device_name', 'window_start', 'record_count', 'octetTotalCount_max', 'octetTotalCount_min', 'octetTotalCount_mean', 'octetTotalCount_median', 'reverseOctetTotalCount_max', 'reverseOctetTotalCount_min', 'reverseOctetTotalCount_mean', 'reverseOctetTotalCount_median', 'packetTotalCount_max', 'packetTotalCount_min', 'packetTotalCount_mean', 'packetTotalCount_median', 'reversePacketTotalCount_max', 'reversePacketTotalCount_min', 'reversePacketTotalCount_mean', 'reversePacketTotalCount_median']
(367, 21)

Samples per device_id:
device_id
00:0c:29:d2:b0:02    94
00:0c:29:ee:e0:7a    94
00:c3:f4:0f:67:73    65
60:14:b3:b1:91:73    44
80:3f:5d:10:17:e1    28
dc:56:e7:5b:61:41    22
d4:dc:cd:b4:26:3e    20
Name: count, dtype: int64

Samples per device_name:
device_name
service_1    94
service_3    94
service_4    65
service_2    44
service_5    28
service_7    22
service_6    20
Name: count, dtype: int64


,source_file,device_id,device_name,window_start,record_count,octetTotalCount_max,octetTotalCount_min,octetTotalCount_mean,octetTotalCount_median,reverseOctetTotalCount_max,...,reverseOctetTotalCount_mean,reverseOctetTotalCount_median,packetTotalCount_max,packetTotalCount_min,packetTotalCount_mean,packetTotalCount_median,reversePacketTotalCount_max,reversePacketTotalCount_min,reversePacketTotalCount_mean,reversePacketTotalCount_median
0,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554323e+09,94,1240.0,70.0,197.957447,186.0,1232.0,...,20.436170,0.0,18,1,2.223404,2.0,16,0,0.244681,0.0
1,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554325e+09,138,17089.0,62.0,354.891304,186.0,47043.0,...,460.376812,0.0,37,1,2.673913,2.0,35,0,0.739130,0.0
2,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554327e+09,128,2750.0,70.0,209.570312,186.0,4221.0,...,45.492188,0.0,21,1,2.242188,2.0,18,0,0.281250,0.0
3,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554329e+09,123,2700.0,73.0,222.560976,186.0,4371.0,...,68.406504,0.0,20,1,2.260163,2.0,16,0,0.235772,0.0
4,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554331e+09,127,2700.0,73.0,220.803150,186.0,5687.0,...,70.330709,0.0,22,1,2.291339,2.0,20,0,0.283465,0.0


In [14]:
import os
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/iot/iot device name/network"
FEATURE_DIR = os.path.join(BASE, "Okui22", "features")
final_out = os.path.join(FEATURE_DIR, "okui22_window_features_all.csv")

df = pd.read_csv(final_out)

print(df.shape)
print(df.columns.tolist())

# 标签用 device_id 更稳
label_col = "device_id"

# 不作为特征的列
drop_cols = ["source_file", "device_id", "device_name", "window_start"]

feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[label_col].copy()

print("num features =", len(feature_cols))
print("feature cols =", feature_cols)
print("\nclass counts:")
print(y.value_counts())

(367, 21)
['source_file', 'device_id', 'device_name', 'window_start', 'record_count', 'octetTotalCount_max', 'octetTotalCount_min', 'octetTotalCount_mean', 'octetTotalCount_median', 'reverseOctetTotalCount_max', 'reverseOctetTotalCount_min', 'reverseOctetTotalCount_mean', 'reverseOctetTotalCount_median', 'packetTotalCount_max', 'packetTotalCount_min', 'packetTotalCount_mean', 'packetTotalCount_median', 'reversePacketTotalCount_max', 'reversePacketTotalCount_min', 'reversePacketTotalCount_mean', 'reversePacketTotalCount_median']
num features = 17
feature cols = ['record_count', 'octetTotalCount_max', 'octetTotalCount_min', 'octetTotalCount_mean', 'octetTotalCount_median', 'reverseOctetTotalCount_max', 'reverseOctetTotalCount_min', 'reverseOctetTotalCount_mean', 'reverseOctetTotalCount_median', 'packetTotalCount_max', 'packetTotalCount_min', 'packetTotalCount_mean', 'packetTotalCount_median', 'reversePacketTotalCount_max', 'reversePacketTotalCount_min', 'reversePacketTotalCount_mean', 'r

In [16]:
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import average_precision_score

# 编码标签
le = LabelEncoder()
y_enc = le.fit_transform(y)

# 分层划分
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc,
    test_size=0.3,
    random_state=42,
    stratify=y_enc
)

print("train shape:", X_train.shape)
print("test shape:", X_test.shape)
print("classes:", list(le.classes_))

# LightGBM
clf = LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

clf.fit(X_train, y_train)

# 预测概率
proba = clf.predict_proba(X_test)
if isinstance(proba, list):
    proba = np.column_stack(proba)

# 多分类 AUCPR
Y_test_bin = label_binarize(y_test, classes=np.arange(len(le.classes_)))
macro_aucpr = average_precision_score(Y_test_bin, proba, average="macro")
weighted_aucpr = average_precision_score(Y_test_bin, proba, average="weighted")

print("macro-AUCPR   =", macro_aucpr)
print("weighted-AUCPR =", weighted_aucpr)

train shape: (256, 17)
test shape: (111, 17)
classes: ['00:0c:29:d2:b0:02', '00:0c:29:ee:e0:7a', '00:c3:f4:0f:67:73', '60:14:b3:b1:91:73', '80:3f:5d:10:17:e1', 'd4:dc:cd:b4:26:3e', 'dc:56:e7:5b:61:41']
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000545 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 520
[LightGBM] [Info] Number of data points in the train set: 256, number of used features: 13
[LightGBM] [Info] Start training from score -1.355523
[LightGBM] [Info] Start training from score -1.355523
[LightGBM] [Info] Start training from score -1.738515
[LightGBM] [Info] Start training from score -2.111190
[LightGBM] [Info] Start training from score -2.600738
[LightGBM] [Info] Start training from score -2.906120
[LightGBM] [Info] Start training from score -2.837127
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

In [17]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd

# 读数据
df = pd.read_csv(final_out)

label_col = "device_id"
drop_cols = ["source_file", "device_id", "device_name", "window_start"]
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[label_col].copy()
groups = df["source_file"].values

# 标签编码
le = LabelEncoder()
y_enc = le.fit_transform(y)

# 按 source_file 分组切分
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y_enc, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y_enc[train_idx], y_enc[test_idx]

print("train files:", sorted(df.iloc[train_idx]["source_file"].unique()))
print("test files :", sorted(df.iloc[test_idx]["source_file"].unique()))
print("train shape:", X_train.shape)
print("test shape :", X_test.shape)

clf = LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)
clf.fit(X_train, y_train)

proba = clf.predict_proba(X_test)
if isinstance(proba, list):
    proba = np.column_stack(proba)

Y_test_bin = label_binarize(y_test, classes=np.arange(len(le.classes_)))
macro_aucpr = average_precision_score(Y_test_bin, proba, average="macro")
weighted_aucpr = average_precision_score(Y_test_bin, proba, average="weighted")

print("Grouped macro-AUCPR   =", macro_aucpr)
print("Grouped weighted-AUCPR =", weighted_aucpr)

train files: ['normal_11_packets.csv', 'normal_12_packets.csv', 'normal_13_packets.csv', 'normal_1_packets.csv', 'normal_2_packets.csv', 'normal_3_packets.csv', 'normal_4_packets.csv', 'normal_7_packets.csv', 'normal_9_packets.csv']
test files : ['normal_10_packets.csv', 'normal_5_packets.csv', 'normal_6_packets.csv', 'normal_8_packets.csv']
train shape: (249, 17)
test shape : (118, 17)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000109 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 529
[LightGBM] [Info] Number of data points in the train set: 249, number of used features: 13
[LightGBM] [Info] Start training from score -1.327798
[LightGBM] [Info] Start training from score -1.327798
[LightGBM] [Info] Start training from score -1.779783
[LightGBM] [Info] Start training from score -2.083466
[LightGBM] [Info] Start training from score -2.809403
[LightGBM] [Info] Start training from score -2.878396


In [22]:
import os
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import average_precision_score

# ========= 1) 读入特征表 =========
BASE = "/content/drive/MyDrive/iot/iot device name/network"
FEATURE_DIR = os.path.join(BASE, "Okui22", "features")
final_out = os.path.join(FEATURE_DIR, "okui22_window_features_all.csv")

df = pd.read_csv(final_out).copy()

# ========= 2) 文件划分 =========
idle_train_files = [f"normal_{i}_packets.csv" for i in range(1, 7)]   # 1-6
idle_test_files  = [f"normal_{i}_packets.csv" for i in range(7, 9)]   # 7-8

active_train_files = [f"normal_{i}_packets.csv" for i in range(9, 12)]   # 9-11
active_test_files  = [f"normal_{i}_packets.csv" for i in range(12, 14)]  # 12-13

train_files = idle_train_files + active_train_files
test_idle_files = idle_test_files
test_active_files = active_test_files
test_mix_files = test_idle_files + test_active_files

# ========= 3) 划出训练/测试 =========
label_col = "device_id"   # 如果你想试 device_name，只改这一行
drop_cols = ["source_file", "device_id", "device_name", "window_start"]
feature_cols = [c for c in df.columns if c not in drop_cols]

train_df = df[df["source_file"].isin(train_files)].copy()
test_idle_df = df[df["source_file"].isin(test_idle_files)].copy()
test_active_df = df[df["source_file"].isin(test_active_files)].copy()
test_mix_df = df[df["source_file"].isin(test_mix_files)].copy()

print("train files:", train_files)
print("test idle files:", test_idle_files)
print("test active files:", test_active_files)

print("\ntrain shape:", train_df.shape)
print("test idle shape:", test_idle_df.shape)
print("test active shape:", test_active_df.shape)
print("test mix shape:", test_mix_df.shape)

# ========= 4) 只保留训练和各测试集共同拥有的类别 =========
common_labels = set(train_df[label_col])
common_labels &= set(test_idle_df[label_col])
common_labels &= set(test_active_df[label_col])
common_labels &= set(test_mix_df[label_col])
common_labels = sorted(common_labels)

train_df = train_df[train_df[label_col].isin(common_labels)].copy()
test_idle_df = test_idle_df[test_idle_df[label_col].isin(common_labels)].copy()
test_active_df = test_active_df[test_active_df[label_col].isin(common_labels)].copy()
test_mix_df = test_mix_df[test_mix_df[label_col].isin(common_labels)].copy()

print("\ncommon labels:", len(common_labels))
print("train class counts:")
print(train_df[label_col].value_counts())

# ========= 5) 编码标签 =========
le = LabelEncoder()
le.fit(common_labels)

train_df["y"] = le.transform(train_df[label_col])
test_idle_df["y"] = le.transform(test_idle_df[label_col])
test_active_df["y"] = le.transform(test_active_df[label_col])
test_mix_df["y"] = le.transform(test_mix_df[label_col])

# ========= 6) 训练 =========
X_train = train_df[feature_cols].copy()
y_train = train_df["y"].values

clf = LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)
clf.fit(X_train, y_train)

# ========= 7) 评估函数 =========
def macro_aucpr_for_subset(sub_df):
    if len(sub_df) == 0:
        return np.nan

    X_sub = sub_df[feature_cols].copy()
    y_sub = sub_df["y"].values

    proba = clf.predict_proba(X_sub)
    if isinstance(proba, list):
        proba = np.column_stack(proba)

    Y_bin = label_binarize(y_sub, classes=np.arange(len(le.classes_)))

    scores = []
    for i in range(len(le.classes_)):
        if Y_bin[:, i].sum() == 0:
            continue
        ap = average_precision_score(Y_bin[:, i], proba[:, i])
        scores.append(ap)

    return float(np.mean(scores)) if len(scores) > 0 else np.nan

idle_aucpr = macro_aucpr_for_subset(test_idle_df)
active_aucpr = macro_aucpr_for_subset(test_active_df)
mix_aucpr = macro_aucpr_for_subset(test_mix_df)

result_table = pd.DataFrame([{
    "Train": "Mix",
    "Test: Idle": idle_aucpr,
    "Test: Active": active_aucpr,
    "Test: Mix": mix_aucpr,
}])

display(result_table)

train files: ['normal_1_packets.csv', 'normal_2_packets.csv', 'normal_3_packets.csv', 'normal_4_packets.csv', 'normal_5_packets.csv', 'normal_6_packets.csv', 'normal_9_packets.csv', 'normal_10_packets.csv', 'normal_11_packets.csv']
test idle files: ['normal_7_packets.csv', 'normal_8_packets.csv']
test active files: ['normal_12_packets.csv', 'normal_13_packets.csv']

train shape: (259, 21)
test idle shape: (45, 21)
test active shape: (63, 21)
test mix shape: (108, 21)

common labels: 5
train class counts:
device_id
00:0c:29:d2:b0:02    65
00:0c:29:ee:e0:7a    65
00:c3:f4:0f:67:73    42
dc:56:e7:5b:61:41    16
d4:dc:cd:b4:26:3e    12
Name: count, dtype: int64
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000065 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 454
[LightGBM] [Info] Number of data points in the train set: 200, number of used features: 13
[LightGBM] [Info] Start training from score -1.1

,Train,Test: Idle,Test: Active,Test: Mix
0,Mix,0.823611,0.823333,0.826435
